# Day 25: Edge Cases and Failure Modes

An edge case is a valid but unusual input. A failure mode is something actually breaking. This notebook covers both: weird-but-valid queries, normalization, timeouts and retries, a circuit breaker, graceful fallbacks, bad index data, a full failure matrix, and one resilient wrapper combining every protection. Everything here runs fast and finishes in a few seconds.

## Weird inputs

Emoji-only, very long, repeated characters, mixed language, and instruction-like text, each with a stated expectation for how RagService should treat it.

In [ ]:
import sys, os, io
if hasattr(sys.stdout, 'buffer'):
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'day-22-production-rag'))
from config import Config
from rag_service import RagService

service = RagService(Config())
for q in ['\U0001F914\U0001F525', 'What is RAG?', '']:
    r = service.query(q)
    print(repr(q)[:40], '->', r.status)


## Normalization

Collapsing whitespace, stripping control characters, Unicode NFKC normalization, and capping repeated characters, all before a query ever reaches retrieval.

In [ ]:
from input_normalization import normalize_query
print(repr(normalize_query('What    is   RAG?\n\n\n')))
print(repr(normalize_query('soooooooooooo good')))
print(repr(normalize_query('\uff37\uff48\uff41\uff54 is RAG?')))


## Timeouts and retries

A soft timeout measures elapsed time and discards a late result. Retries are capped at 2, with exponential backoff, and the whole demo stays under 2 seconds.

In [ ]:
from timeouts_and_retries import FakeSlowRetriever, call_with_retries
import time

start = time.perf_counter()
retriever = FakeSlowRetriever(delays_seconds=[0.5, 0.05])
outcome = call_with_retries(retriever.retrieve, 'What is RAG?')
print('attempts:', outcome['attempts'], 'timed_out:', outcome['timed_out'])
print('elapsed so far:', round(time.perf_counter() - start, 2), 's')


## Circuit breaker

Three states: closed, open (fail fast), half-open (one probe call). Opens after 3 consecutive failures, recovers after a short cooldown.

In [ ]:
from circuit_breaker import CircuitBreaker, CircuitOpenError

breaker = CircuitBreaker(failure_threshold=2, cooldown_seconds=0.1)
def always_fails():
    raise RuntimeError('simulated failure')

for i in range(3):
    try:
        breaker.call(always_fails)
    except (RuntimeError, CircuitOpenError) as e:
        print(i, type(e).__name__, breaker.state)


## Graceful fallbacks

If the ranker fails, fall back to retrieval order (status degraded). If the generator fails, return top source snippets instead (status partial).

In [ ]:
import graceful_fallbacks
graceful_fallbacks.main()


## Bad index data

Documents with empty text, None, duplicate ids, and absurdly long text are skipped with a recorded reason, while valid documents still index and serve correctly.

In [ ]:
import bad_index_data
bad_index_data.main()


## Failure matrix

Every failure mode in one table: expected status, actual status, whether the message was safe, and PASS or FAIL.

In [ ]:
import failure_matrix
failure_matrix.main()


## Resilient service comparison

The same five requests through the plain RagService and the resilient wrapper, side by side.

In [ ]:
import resilient_service
resilient_service.main()


## Key takeaways

- An edge case is valid-but-unusual input, a failure mode is something actually breaking, and both deserve tests written on purpose, not discovered by accident.
- Normalizing input early means every later stage works with the same cleaned-up text instead of each reinventing its own cleanup.
- A soft timeout treats a result that arrived too late the same as a failure, since a caller already moved on.
- Retries need a cap and backoff, and are only safe for idempotent work.
- A circuit breaker protects both the caller, who stops wasting time on a doomed call, and the struggling dependency, which gets a real cooldown.
- A graceful fallback gives a user something real and honestly labeled, degraded or partial, instead of nothing at all.
- A failure matrix run every release turns 'I think this still works' into a visible PASS or FAIL per failure mode.